# 什么是 LU 分解？

> 写给学完了《数学分析》《高等代数》的你。我们从你已经会的东西——**高斯消元**——出发，一步步推出 LU 分解，全程不跳步。

先记住一句话：**LU 分解，就是把“解方程组时的高斯消元过程”，拆成一个下三角矩阵 L 和一个上三角矩阵 U 的乘积**：

$$A = LU$$

- $L$ = **L**ower（下三角），对角线全是 1
- $U$ = **U**pper（上三角）

> 建议：每个代码格都亲手点一下运行，看输出，再读下一段。慢慢来。


## 一、从高斯消元说起

解方程组 $Ax = b$，你在高代里学过的标准做法是：对增广矩阵做**初等行变换**，把系数矩阵 $A$ 化成上三角（阶梯形），再回代求解。

关键点（高代知识）：**每一类初等行变换，都等价于左乘一个初等矩阵**。

- “第 2 行减去 3 倍第 1 行”  ⇔  左乘某个初等矩阵 $E$
- “交换两行”  ⇔  左乘一个置换矩阵 $P$

所以“消元到上三角”这件事，用矩阵语言写就是：

$$E_k \cdots E_2 \, E_1 \, A = U$$

其中 $U$ 是上三角矩阵，符号 $\cdots$ 表示连乘。


## 二、把消元“搬到右边”，就得到 L

上面的式子两边同时左乘 $E_1^{-1} E_2^{-1} \cdots E_k^{-1}$，得到：

$$A = (E_1^{-1} E_2^{-1} \cdots E_k^{-1}) \cdot U$$

高代里有一个结论：**这些初等矩阵的逆乘在一起，恰好是一个下三角矩阵**（对角元还是 1）。把这个乘积记作 $L$，就得到：

$$A = LU$$

这就是 **LU 分解**。

> 直觉：$U$ 记录“消元消到最后的样子”；$L$ 记录“每一步消元用了多大的倍数（乘子）”。两者合起来，完整保存了高斯消元的全过程。


## 三、为什么要费这个劲？——解方程变简单

有了 $A = LU$，解 $Ax = b$ 就拆成两步：

1. **前代**（forward substitution）：解 $Ly = b$
2. **回代**（back substitution）：解 $Ux = y$

为什么能这样拆？因为

$$Ax = LUx = L(Ux) = b$$

令 $y = Ux$，先解出 $y$（前代），再解 $x$（回代）。

好处：

- 解三角方程特别快，代价是 $O(n^2)$；直接高斯消元是 $O(n^3)$。
- **L、U 算一次，可以反复用**：以后要解 $Ax = b_1, \, Ax = b_2, \dots$（矩阵没变、右端变了），不用重新分解，直接套用同一组 L、U。


## 四、一个 3×3 的具体例子（先读，再跑下面的代码）

下面几个代码格会做三件事，你从上往下逐格运行：

1. 造一个 3×3 矩阵 $A$ 和向量 $b$；
2. 用“不带主元”的算法把它分解成 $L$ 和 $U$；
3. 验证 $L \cdot U$ 确实等于原来的 $A$。

跑之前先记住一句话：**$L$ 里那些非 1 的数，就是消元乘子**（“第 i 行要减去多少倍的第 k 行”）。


In [1]:
import numpy as np

# 一个 3x3 例子（选整数矩阵，算起来清楚）
A = np.array([
    [2.0, 1.0, 1.0],
    [4.0, 3.0, 3.0],
    [8.0, 7.0, 9.0],
])

# 右端向量（随意给一个）
b = np.array([1.0, 2.0, 3.0])

print('A =')
print(A)
print('b =', b)


A =
[[2. 1. 1.]
 [4. 3. 3.]
 [8. 7. 9.]]
b = [1. 2. 3.]


## 五、手把手：这个例子的 L 和 U 是怎么来的

消元第一步（消掉第 1 列主元下面的元素）：

- 第 2 行要减去 $a_{21}/a_{11} = 4/2 = 2$ 倍第 1 行 → 乘子 $l_{21} = 2$
- 第 3 行要减去 $a_{31}/a_{11} = 8/2 = 4$ 倍第 1 行 → 乘子 $l_{31} = 4$

这些乘子会填进 $L$ 的第 1 列。

消元第二步（消第 2 列主元下面的元素），又会得到一个乘子填进 $L$ 的第 2 列……

**规律**：$L$ 的第 $k$ 列，装的正是“第 $k$ 步消元时各行用到的乘子”。运行下一格，你会看到 $L$ 里的 2 和 4 正好对上。


In [2]:
def lu_no_pivot(A):
    '''不带主元的 LU 分解：A = L @ U（L 的对角元全为 1）'''
    A = A.astype(float).copy()
    n = A.shape[0]
    L = np.eye(n)      # 先让 L 是单位矩阵，之后只填下三角的乘子
    U = A.copy()       # U 从 A 出发，慢慢把下三角消成 0
    for k in range(n):                # k 是当前“主元”所在的行/列
        for i in range(k + 1, n):     # 处理主元下面的每一行 i
            L[i, k] = U[i, k] / U[k, k]       # ① 乘子 l_ik
            U[i, k:] -= L[i, k] * U[k, k:]    # ② 行 i 减去 l_ik 倍行 k
            U[i, k] = 0.0                      # ③ 把下三角位置显式清零
    return L, U

L, U = lu_no_pivot(A)
print('L =')
print(L)
print('U =')
print(U)
print('L @ U =')
print(L @ U)
print('和原来的 A 相等吗？', np.allclose(L @ U, A))


L =
[[1. 0. 0.]
 [2. 1. 0.]
 [4. 3. 1.]]
U =
[[2. 1. 1.]
 [0. 1. 1.]
 [0. 0. 2.]]
L @ U =
[[2. 1. 1.]
 [4. 3. 3.]
 [8. 7. 9.]]
和原来的 A 相等吗？ True


## 六、三角方程怎么解：前代与回代

**前代**（解 $Ly = b$，$L$ 下三角）：$L$ 的第 1 行只有一个未知数 $y_1$，直接解出；代入第 2 行解出 $y_2$……**从上往下**逐个解，所以叫“前代”。

**回代**（解 $Ux = y$，$U$ 上三角）：$U$ 的最后一行只有一个未知数 $x_n$，直接解出；再往上逐个解……**从下往上**，所以叫“回代”。

因为 $L$ 的对角元都是 1，前代连除法都不用做。运行下一格看结果。


In [ ]:
def forward_sub(L, b):
    '''前代：解 L y = b（L 是对角元为 1 的下三角）'''
    n = len(b)
    y = np.zeros(n)
    for i in range(n):
        y[i] = b[i] - L[i, :i] @ y[:i]   # y_i = b_i 减去前面已求出的加权和
    return y

def back_sub(U, y):
    '''回代：解 U x = y（U 是上三角）'''
    n = len(y)
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - U[i, i + 1:] @ x[i + 1:]) / U[i, i]   # 最后除以对角元
    return x

y = forward_sub(L, b)          # 第一步：前代
x = back_sub(U, y)             # 第二步：回代

x_ref = np.linalg.solve(A, b)  # NumPy 给的“标准答案”
print('我们自己算出的 x =', x)
print('NumPy 参考解     =', x_ref)
print('两者的差（应接近 0）=', np.linalg.norm(x - x_ref))


## 七、主元（pivot）与它的陷阱

前面代码里那个被除数 `U[k,k]` 就叫**主元**（pivot）。

如果某一步主元**非常小**，乘子

$$l_{ik} = u_{ik} / u_{kk}$$

就会**非常大**，误差被急剧放大——这就是不带主元 LU 分解会“崩”的根源。

经典例子（2×2 矩阵）：

```text
A = [ [eps, 1],
      [1,   1] ]       其中 eps = 10^(-12)
```

这个矩阵其实**很健康**（不病态），但它的第一个主元是 eps，于是乘子

$$l_{21} = 1 / eps = 10^{12}$$

巨大无比。运行下一格，你会看到灾难性的误差。


In [ ]:
eps = 1e-12

A_bad = np.array([
    [eps, 1.0],
    [1.0, 1.0],
])

# 让精确解是全 1，反推出 b
b_bad = A_bad @ np.array([1.0, 1.0])

L, U = lu_no_pivot(A_bad)
y = forward_sub(L, b_bad)
x_no = back_sub(U, y)

x_ref = np.linalg.solve(A_bad, b_bad)

print('不带主元算出的 x =', x_no)
print('参考解           =', x_ref)
print('相对误差         =', np.linalg.norm(x_no - x_ref) / np.linalg.norm(x_ref))


## 八、解法：部分主元法（partial pivoting）

思路很简单：**每一步消元前，先在这一列里挑一个绝对值最大的元素当主元，把它所在的行换到当前行**。

用高代的话说，就是先左乘一个置换矩阵 $P$（记录行交换），再对 $PA$ 做不带主元的 LU 分解：

$$PA = LU$$

这样一来，每一步的乘子都满足 $|l_{ik}| \le 1$（因为主元是列里最大的，除下来不会超过 1），误差就不会被放大。

> 直观理解：交换两行不改变方程组的解（只是把方程的顺序换一下），但能保证我们每次都用“最结实”的那个数当除数。


In [ ]:
def lu_partial_pivot(A):
    '''带部分主元的 LU 分解：P @ A = L @ U'''
    A = A.astype(float).copy()
    n = A.shape[0]
    U = A.copy()
    L = np.eye(n)
    P = np.eye(n)          # 置换矩阵，记录行交换
    for k in range(n):
        # 在当前列第 k 行往下找绝对值最大的元素，记下它的行号 p
        p = int(np.argmax(np.abs(U[k:, k]))) + k
        if p != k:
            U[[k, p], :] = U[[p, k], :]   # 交换 U 的第 k、p 行
            P[[k, p], :] = P[[p, k], :]   # 同步交换 P
            L[[k, p], :k] = L[[p, k], :k] # 交换 L 里已算好的乘子
        for i in range(k + 1, n):
            L[i, k] = U[i, k] / U[k, k]
            U[i, k:] -= L[i, k] * U[k, k:]
            U[i, k] = 0.0
    return P, L, U

# 用部分主元法重新解刚才那个“崩掉”的方程
P, L, U = lu_partial_pivot(A_bad)
y = forward_sub(L, P @ b_bad)   # 注意：右端也要跟着做同样的行交换
x_pp = back_sub(U, y)

print('部分主元算出的 x =', x_pp)
print('参考解           =', x_ref)
print('相对误差         =', np.linalg.norm(x_pp - x_ref) / np.linalg.norm(x_ref))
print('P =（记录行交换）')
print(P)


## 九、小结与名词表

**一句话总结**：LU 分解把高斯消元存成 $A = LU$；不带主元时遇到小主元会放大误差；部分主元法通过行交换让乘子不超过 1，保证数值稳定。

| 名词 | 含义 |
|---|---|
| LU 分解 | 把 $A$ 写成下三角 $L$ × 上三角 $U$ |
| Doolittle 形式 | $L$ 对角元全为 1 的那一种 |
| 主元 pivot | 消元时对角线上的除数 $u_{kk}$ |
| 乘子 multiplier | $l_{ik} = u_{ik}/u_{kk}$，填入 $L$ |
| 前代 / 回代 | 分别解 $Ly = b$ 和 $Ux = y$ |
| 部分主元 partial pivoting | 每步选列中最大元素当主元并换行 |
| 置换矩阵 $P$ | 记录行交换，满足 $PA = LU$ |
| 条件数 κ(A) | 衡量矩阵“病态程度”（可先跳过）|

学到这里，再回头看工作区里的 `LU.py`，你会发现它每一行都在做今天讲的事。想深入，去读 `finish.pdf` 里的数学分析报告。
